# 03. Experimento controlado

Un experimento es un conjunto pequeño de configuraciones que comparten todo salvo la variable que se quiere estudiar. Este notebook corre un experimento y arma la evidencia.

Para otro experimento se duplica el notebook y se cambia la celda de parámetros.

## 0. Hipótesis

Se escribe antes de correr.

**Observación de partida:** *por escribir*

**Hipótesis:** *por escribir*

**Qué espero ver si es cierta:** *por escribir*

**Qué vería si no lo es:** *por escribir*

## 1. Parámetros

`CONFIG_FILES` son archivos de `configs/` que pertenecen al mismo experimento. `REFERENCE` es el nombre de la configuración contra la que se comparan las demás.

In [ ]:
CONFIG_FILES = ["res_064.toml", "res_128.toml", "res_224.toml"]
REFERENCE = "res128"
SEEDS = [11, 22, 33]

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from nubes import figuras
from nubes.config import RunConfig, diff
from nubes.datos import load_manifest
from nubes.experimento import build_model, evaluate_test, run_seeds
from nubes.inspeccion import receptive_field
from nubes.metricas import symmetric_confusions
from nubes.modelo import count_parameters
from nubes.resumen import aggregate, per_class_recall, runs_table, to_markdown, total_confusion
from nubes.utilidades import PROJECT_ROOT

configs = {
    c.name: c for c in (RunConfig.from_toml(PROJECT_ROOT / "configs" / f) for f in CONFIG_FILES)
}
reference = configs[REFERENCE]
assert len({c.experiment for c in configs.values()}) == 1, (
    "Las configuraciones son de experimentos distintos"
)
_, stats = load_manifest(reference.data_dir)
CLASSES = stats["classes"]
print("experimento:", reference.experiment, "| configuraciones:", list(configs))

## 2. Qué cambia y qué queda fijo

La primera tabla lista los campos que difieren de la referencia. Todo lo que no aparece es igual. La segunda muestra lo que cada configuración implica para la red.

In [ ]:
changes = pd.DataFrame(
    {
        name: {k: after for k, (_, after) in diff(reference, c).items()}
        for name, c in configs.items()
    }
).T.fillna("igual")
display(changes if len(changes.columns) else "Ninguna configuración difiere de la referencia")

rows = {}
for name, c in configs.items():
    model = build_model(c, len(CLASSES))
    last = receptive_field(model, c.image_size).iloc[-1]
    rows[name] = {
        "image_size": c.image_size,
        "último mapa": int(last["map_size"]),
        "campo receptivo": int(last["rf"]),
        "fracción de la imagen": round(float(last["fraction"]), 3),
        "parámetros": count_parameters(model),
    }
pd.DataFrame(rows).T

## 3. Corridas

In [ ]:
results = {name: run_seeds(c, SEEDS) for name, c in configs.items()}

## 4. Resultados en validation

Media y desviación entre semillas. Una diferencia entre configuraciones menor que la dispersión entre semillas no se puede atribuir a la variable.

In [ ]:
table = runs_table(results)
summary = aggregate(table)
display(table.round(4))
display(summary.round(4))
print(to_markdown(summary))

In [ ]:
figuras.plot_seed_points(table, "val_macro_f1")
figuras.plot_seed_points(table, "gap")
figuras.plot_comparison(results);

**Observación:**

*por escribir*

## 5. Por clase

La hipótesis casi nunca predice lo mismo para todas las clases. La primera tabla es el recall medio. La segunda es el cambio frente a la referencia y la tercera la desviación entre semillas, que da la escala para leer ese cambio.

In [ ]:
recall = per_class_recall(results, CLASSES)
recall_std = per_class_recall(results, CLASSES, stat="std")
figuras.plot_recall_by_class(recall)
display(recall.round(3))
display(recall.sub(recall[REFERENCE], axis=0).drop(columns=REFERENCE).round(3))
recall_std.round(3)

In [ ]:
reference_confusion = total_confusion(results[REFERENCE])
for name, runs in results.items():
    if name == REFERENCE:
        continue
    change = total_confusion(runs) - reference_confusion
    figuras.plot_matrix(
        change,
        CLASSES,
        fmt="+d",
        cmap="RdBu_r",
        diverging=True,
        title=f"{name} menos {REFERENCE}. Suma de {len(SEEDS)} semillas",
    )
    display(symmetric_confusions(total_confusion(runs), CLASSES, k=6))

**Observación:**

*por escribir*

## 6. Costo

In [ ]:
table.groupby("config", sort=False)[["seconds_per_epoch", "best_epoch", "parameters"]].mean().round(
    1
)

## 7. Veredicto

**Qué se observó:** *por escribir*

**Qué se interpreta:** *por escribir*

**La hipótesis se sostiene, no se sostiene o no se puede decidir:** *por escribir*

**Qué limita esta conclusión:** *por escribir*

## 8. Evaluación final en test

Solo al final del proyecto y solo para la configuración elegida con validation. Cada corrida se puede evaluar una vez. El resultado queda en `test.json` dentro de su carpeta.

In [ ]:
EVALUATE_TEST = False
FINAL = REFERENCE

if EVALUATE_TEST:
    reports = [evaluate_test(r.run_dir) for r in results[FINAL]]
    final = pd.DataFrame(
        {
            "seed": [r.config.seed for r in results[FINAL]],
            "val_acc": [r.val.accuracy for r in results[FINAL]],
            "test_acc": [t.accuracy for t in reports],
            "val_macro_f1": [r.val.macro_f1 for r in results[FINAL]],
            "test_macro_f1": [t.macro_f1 for t in reports],
        }
    )
    display(final.round(4))
    figuras.plot_matrix(
        sum(np.asarray(t.confusion) for t in reports),
        CLASSES,
        normalize=True,
        title=f"{FINAL}. Test, {len(reports)} semillas",
    )